# Lab 3: Probability, Statistics & Significance in Bioinformatics

This notebook follows the 90-minute lab exercises: from sampling and t-tests to ANOVA, BH FDR and bootstrapping — all on **real data with real sample groups**.

**Dataset — GEO GSE10072** (Landi *et al.*, 2008, *PLoS ONE*): gene expression in **lung adenocarcinoma** and **non-tumor lung tissue** from **never, former and current smokers**.
* Platform: Affymetrix HG-U133A (GPL96), 22 283 probe sets.
* 107 samples: **58 tumor** and **49 non-tumor** tissues.
* Values are normalised **log2 intensities** (continuous values, *not* read counts).

**Environment:** Python, `GEOparse`, `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`, `seaborn`, `scikit-learn`. Internet access is needed on the first run (the data are then cached in the working directory).

In [1]:
pip install GEOparse

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import GEOparse

sns.set(style='whitegrid', context='notebook', rc={'figure.figsize': (8, 5)})
rng = np.random.default_rng(42)
np.random.seed(42)
pd.set_option('display.max_colwidth', 120)

print('Libraries imported. numpy, pandas, scipy, statsmodels, matplotlib, seaborn, GEOparse')

Libraries imported. numpy, pandas, scipy, statsmodels, matplotlib, seaborn, GEOparse


## Loading the data

`GEOparse` downloads the full SOFT file of the series. It contains three things we need:
1. the **expression values** of every sample (`gse.pivot_samples('VALUE')`),
2. the **sample metadata** (`gse.gsms[...].metadata`) — this is where the groups come from,
3. the **platform annotation** (`gse.gpls['GPL96'].table`) — probe ID → gene symbol.

In [ ]:
gse_id = 'GSE10072'
gse = GEOparse.get_GEO(geo=gse_id, destdir='.', silent=True)   # cached after the first download

expr = gse.pivot_samples('VALUE')                 # rows = probes, columns = samples (GSM IDs)
expr = expr.apply(pd.to_numeric, errors='coerce').dropna()
print(f'Expression matrix: {expr.shape[0]} probes x {expr.shape[1]} samples')
print(f'Value range: {expr.values.min():.2f} - {expr.values.max():.2f}')

In [ ]:
# Sample metadata: one row per GSM
rows = {}
for gsm_id, gsm in gse.gsms.items():
    m = gsm.metadata
    rows[gsm_id] = {'title': m.get('title', [''])[0],
                    'source': m.get('source_name_ch1', [''])[0],
                    'characteristics': ' | '.join(m.get('characteristics_ch1', []))}
meta_raw = pd.DataFrame.from_dict(rows, orient='index')
print('Example metadata (look at how groups are described):')
meta_raw.head(8)

In [ ]:
# Turn the free-text descriptions into group labels
text = (meta_raw['title'] + ' ' + meta_raw['source'] + ' ' + meta_raw['characteristics']).str.lower()

# Tissue: check "normal / non-tumor" FIRST, because "non-tumor" also contains the word "tumor"
is_normal = text.str.contains(r'non[- ]?tumou?r|normal|non[- ]?involved|uninvolved')
is_tumor = text.str.contains(r'tumou?r|adenocarcinoma|carcinoma|cancer')
tissue = pd.Series('unknown', index=text.index)
tissue[is_tumor & ~is_normal] = 'tumor'
tissue[is_normal] = 'normal'

# Smoking status: a label is assigned only if exactly one pattern matches
patterns = {'never': r'never|non[- ]?smok', 'former': r'former|ex[- ]?smok', 'current': r'current'}
hits = pd.DataFrame({k: text.str.contains(p) for k, p in patterns.items()})
smoking = pd.Series(np.nan, index=text.index, dtype=object)
one = hits.sum(axis=1) == 1
smoking[one] = hits[one].idxmax(axis=1)

meta = pd.DataFrame({'tissue': tissue, 'smoking': smoking}).loc[expr.columns]
meta['smoking'] = pd.Categorical(meta['smoking'], categories=['never', 'former', 'current'])

print(meta['tissue'].value_counts().to_string(), '\n')
print(pd.crosstab(meta['tissue'], meta['smoking'], dropna=False, margins=True))

# Sanity check against the study description (58 tumor, 49 non-tumor)
counts = meta['tissue'].value_counts()
if counts.get('tumor', 0) != 58 or counts.get('normal', 0) != 49:
    print('\nUnique descriptions found in the metadata:')
    print(text.str.replace(r'gsm\d+', '', regex=True).value_counts().head(20))
    raise ValueError('Tissue labels do not match 58 tumor / 49 normal - adjust the regex patterns above.')

In [ ]:
# Probe -> gene symbol (from the platform annotation)
gpl = list(gse.gpls.values())[0].table
sym_col = 'Gene Symbol' if 'Gene Symbol' in gpl.columns else [c for c in gpl.columns if 'symbol' in c.lower()][0]
probe_symbol = (gpl.set_index('ID')[sym_col].astype(str)
                .str.split(' /// ').str[0].replace({'nan': np.nan}))

def sym(probe):
    s = probe_symbol.get(probe, np.nan)
    return s if isinstance(s, str) else probe

def probe_for(symbol):
    # Return the probe of a gene with the highest mean expression (a gene can have several probes)
    ids = probe_symbol[probe_symbol == symbol].index.intersection(expr.index)
    if len(ids) == 0:
        raise KeyError(f'{symbol} not found on the platform')
    return expr.loc[ids].mean(axis=1).idxmax()

# Remove Affymetrix control probes
expr = expr.loc[~expr.index.astype(str).str.startswith('AFFX')]

tumor_cols = meta.index[meta['tissue'] == 'tumor']
normal_cols = meta.index[meta['tissue'] == 'normal']
print(f'Probes used: {expr.shape[0]}; tumor samples: {len(tumor_cols)}; normal samples: {len(normal_cols)}')

## Exercise 1 — Setup & Warm-up

Explore the dataset: summary statistics, distribution of every sample (quality control), a global view (PCA), and one gene in both groups.

## Exercise 2 — Sampling

Sampling distribution (three values) of the mean on control samples.

## Exercise 3 — Hypothesis Testing: t-test

H₀: mean expression of the gene is the same in **tumor** and **normal** tissue.
We use Welch's t-test (does not assume equal variances) and the Mann–Whitney U test for comparison.

## Exercise 4 — ANOVA

Does expression differ between **never, former and current smokers**?

In [ ]:
gene_a = probe_for('NEK2')

tum = meta.loc[tumor_cols].dropna(subset=['smoking'])
df_anova = tum.assign(value=expr.loc[gene_a, tum.index])
groups = [df_anova.loc[df_anova['smoking'] == s, 'value'].values for s in ['never', 'former', 'current']]
print('Tumor samples per group:', {s: len(g) for s, g in zip(['never', 'former', 'current'], groups)})

## Exercise 5 — Multiple Testing & FDR

Test **every probe** (tumor vs normal, Welch t-test), then correct with **Benjamini–Hochberg** (FDR) and **Bonferroni** (FWER). Plot results on volcano plot. As a contrast we repeat the analysis with **randomly shuffled labels**.

In [ ]:
T = expr[tumor_cols].values
N = expr[normal_cols].values

## Exercise 6 — DGE Mini-Analysis (tumor vs normal)
Select the 30 most significant genes (FDR < 0.05, |log2FC| ≥ 1) and perform hierarchical clustering of genes and samples, visualised as a heatmap of z-scored expression with tumor/normal samples marked. Do the samples cluster by tissue?